# Poke Agent Unified Run

Run this notebook end-to-end.

- On Kaggle/Linux with CABT `cg-lib` available, it can generate rollout data.
- On this Mac, it uses existing rollout JSONL data and trains with Torch on Apple Silicon MPS.
- It does not submit to the competition leaderboard.


In [ ]:
from __future__ import annotations

import glob
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists() and (ROOT.parent / "requirements.txt").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)

print("repo", ROOT)
print("python", sys.version.split()[0])
print("torch", torch.__version__)


In [ ]:
def torch_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = torch_device()
print("device", DEVICE)


In [ ]:
def find_cg_lib() -> str | None:
    candidates: list[str] = []
    if os.environ.get("CG_LIB_PATH"):
        candidates.append(os.environ["CG_LIB_PATH"])
    candidates.extend(glob.glob("/kaggle/input/**/cg-lib", recursive=True))
    candidates.extend(glob.glob(str(ROOT / "kaggle/input/**/cg-lib"), recursive=True))
    return candidates[0] if candidates else None

CG_LIB_PATH = find_cg_lib()
CG_AVAILABLE = False
CG_ERROR = None
if CG_LIB_PATH:
    sys.path.append(CG_LIB_PATH)
    try:
        from cg.game import battle_finish, battle_select, battle_start
        from cg.api import to_observation_class
        CG_AVAILABLE = True
    except Exception as exc:
        CG_ERROR = repr(exc)

print("cg_lib_path", CG_LIB_PATH)
print("cg_available", CG_AVAILABLE)
if CG_ERROR:
    print("cg_error", CG_ERROR)


In [ ]:
SAMPLE_DECK = [
    721, 721, 722, 722, 722, 722, 723, 723, 723, 723,
    1092, 1121, 1121, 1145, 1145, 1163, 1163,
    1219, 1219, 1219, 1219, 1227, 1227, 1227, 1227,
    1262, 1262,
    3, 3, 3, 3, 3, 3, 3, 3, 3,
    3, 3, 3, 3, 3, 3, 3, 3, 3,
    3, 3, 3, 3, 3, 3, 3, 3, 3,
    3, 3, 3, 3, 3, 3,
]

def read_deck() -> list[int]:
    for path in [ROOT / "submission/deck.csv", ROOT / "deck.csv", Path("/kaggle_simulations/agent/deck.csv")]:
        if path.exists():
            deck = [int(line.strip()) for line in path.read_text().splitlines() if line.strip()]
            if len(deck) != 60:
                raise ValueError(f"{path} must contain 60 card IDs")
            return deck
    return SAMPLE_DECK

DECK = read_deck()
print("deck cards", len(DECK))


In [ ]:
def random_agent(obs_dict: dict) -> list[int]:
    obs = to_observation_class(obs_dict)
    options = list(range(len(obs.select.option)))
    return random.sample(options, min(obs.select.maxCount, len(options)))


def features_from_observation(obs: dict) -> list[float]:
    current = obs.get("current") or {}
    players = current.get("players") or [{}, {}]
    p0 = players[0] if len(players) > 0 else {}
    p1 = players[1] if len(players) > 1 else {}
    select = obs.get("select") or {}
    return [
        float(current.get("turn", 0)),
        float(current.get("yourIndex", 0)),
        float(p0.get("deckCount", 0)),
        float(p0.get("handCount", 0)),
        float(len(p0.get("bench", []))),
        float(p1.get("deckCount", 0)),
        float(p1.get("handCount", 0)),
        float(len(p1.get("bench", []))),
        float(len(select.get("option", []))),
        float(select.get("maxCount", 0)),
    ]


def play_episode(episode: int, max_steps: int = 300) -> list[dict]:
    rows = []
    obs, start_data = battle_start(DECK, DECK)
    if start_data.errorPlayer >= 0:
        raise ValueError(f"deck error type={start_data.errorType} player={start_data.errorPlayer}")
    try:
        step = 0
        while obs["current"]["result"] < 0 and step < max_steps:
            rows.append({
                "episode": episode,
                "step": step,
                "features": features_from_observation(obs),
                "player": int(obs["current"]["yourIndex"]),
            })
            obs = battle_select(random_agent(obs))
            step += 1
        result = int(obs["current"]["result"])
        for row in rows:
            row["value"] = 0.0 if result == 2 else (1.0 if row["player"] == result else -1.0)
        return rows
    finally:
        battle_finish()


In [ ]:
GENERATE_EPISODES = int(os.environ.get("CABT_EPISODES", "3" if CG_AVAILABLE else "0"))
GENERATED_PATH = ROOT / "data/notebook_rollouts.jsonl"

if CG_AVAILABLE and GENERATE_EPISODES > 0:
    GENERATED_PATH.parent.mkdir(parents=True, exist_ok=True)
    rows = []
    for episode in range(GENERATE_EPISODES):
        rows.extend(play_episode(episode))
    with GENERATED_PATH.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, separators=(",", ":")) + "\n")
    print(f"generated {len(rows)} rows -> {GENERATED_PATH}")
else:
    print("skipping CABT generation in this runtime")


In [ ]:
DATA_CANDIDATES = [
    ROOT / "data/notebook_rollouts.jsonl",
    ROOT / "data/kaggle-output/data/cabt_rollouts.jsonl",
    ROOT / "data/container-mp-smoke.jsonl",
    ROOT / "data/container-smoke.jsonl",
]

TRANSITION_CLASSES = 8


def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def build_training_arrays(rows: list[dict]) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    by_episode: dict[int, list[dict]] = {}
    for row in rows:
        by_episode.setdefault(int(row["episode"]), []).append(row)
    for episode_rows in by_episode.values():
        episode_rows.sort(key=lambda row: int(row["step"]))

    xs = []
    values = []
    transition_targets = []
    next_features = []
    terminal_mask = []

    for episode_rows in by_episode.values():
        for idx, row in enumerate(episode_rows):
            features = np.array(row["features"], dtype=np.float32)
            value = float(row["value"])
            if idx + 1 < len(episode_rows):
                next_row = episode_rows[idx + 1]
                next_feature = np.array(next_row["features"], dtype=np.float32)
                delta = next_feature - features
                # Coarse pseudo-policy target from observed transition shape.
                transition_class = int(abs(delta).argmax()) % TRANSITION_CLASSES
                is_terminal = 0.0
            else:
                next_feature = features.copy()
                transition_class = TRANSITION_CLASSES - 1
                is_terminal = 1.0

            xs.append(features)
            values.append(value)
            transition_targets.append(transition_class)
            next_features.append(next_feature)
            terminal_mask.append(is_terminal)

    return (
        np.stack(xs).astype(np.float32),
        np.array(values, dtype=np.float32),
        np.array(transition_targets, dtype=np.int64),
        np.stack(next_features).astype(np.float32),
        np.array(terminal_mask, dtype=np.float32),
    )


DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    print("No rollout data found. Using synthetic smoke data so Run All still completes.")
    rng = np.random.default_rng(7)
    x_np = rng.normal(size=(128, 10)).astype(np.float32)
    y_np = np.tanh(x_np[:, 0] * 0.1 + x_np[:, 2] * 0.03 - x_np[:, 5] * 0.03).astype(np.float32)
    transition_np = rng.integers(0, TRANSITION_CLASSES, size=(128,), dtype=np.int64)
    next_x_np = (x_np + rng.normal(scale=0.1, size=x_np.shape)).astype(np.float32)
    terminal_np = np.zeros((128,), dtype=np.float32)
else:
    rows = load_jsonl(DATA_PATH)
    x_np, y_np, transition_np, next_x_np, terminal_np = build_training_arrays(rows)
    print(f"loaded {len(rows)} rows from {DATA_PATH}")

feature_mean_np = x_np.mean(axis=0, keepdims=True)
feature_std_np = x_np.std(axis=0, keepdims=True) + 1e-6
x_norm_np = (x_np - feature_mean_np) / feature_std_np
next_x_norm_np = (next_x_np - feature_mean_np) / feature_std_np

x = torch.tensor(x_norm_np, device=DEVICE)
y = torch.tensor(y_np, device=DEVICE)
transition_target = torch.tensor(transition_np, device=DEVICE)
next_x = torch.tensor(next_x_norm_np, device=DEVICE)
terminal = torch.tensor(terminal_np, device=DEVICE)

print("x", tuple(x.shape), "value", tuple(y.shape), "transition", tuple(transition_target.shape))


In [ ]:
class TransformerRLModel(torch.nn.Module):
    def __init__(self, input_dim: int, policy_dim: int, d_model: int = 64, nhead: int = 4, num_layers: int = 2):
        super().__init__()
        self.input_dim = input_dim
        self.token_proj = torch.nn.Linear(1, d_model)
        self.feature_embed = torch.nn.Embedding(input_dim, d_model)
        encoder_layer = torch.nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = torch.nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = torch.nn.LayerNorm(d_model)
        self.value_head = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_model),
            torch.nn.GELU(),
            torch.nn.Linear(d_model, 1),
        )
        self.policy_head = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_model),
            torch.nn.GELU(),
            torch.nn.Linear(d_model, policy_dim),
        )
        self.next_feature_head = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_model),
            torch.nn.GELU(),
            torch.nn.Linear(d_model, input_dim),
        )
        self.uncertainty_head = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_model),
            torch.nn.GELU(),
            torch.nn.Linear(d_model, 1),
        )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        feature_ids = torch.arange(self.input_dim, device=x.device)
        tokens = self.token_proj(x.unsqueeze(-1)) + self.feature_embed(feature_ids).unsqueeze(0)
        encoded = self.encoder(tokens)
        return self.norm(encoded.mean(dim=1))

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        pooled = self.encode(x)
        return {
            "value": self.value_head(pooled).squeeze(-1),
            "policy_logits": self.policy_head(pooled),
            "next_features": self.next_feature_head(pooled),
            "log_variance": self.uncertainty_head(pooled).squeeze(-1).clamp(-5.0, 5.0),
        }


model = TransformerRLModel(x.shape[1], TRANSITION_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
value_loss_fn = torch.nn.MSELoss()
policy_loss_fn = torch.nn.CrossEntropyLoss()
dynamics_loss_fn = torch.nn.SmoothL1Loss(reduction="none")

VALUE_WEIGHT = float(os.environ.get("LOSS_VALUE_WEIGHT", "1.0"))
POLICY_WEIGHT = float(os.environ.get("LOSS_POLICY_WEIGHT", "0.35"))
DYNAMICS_WEIGHT = float(os.environ.get("LOSS_DYNAMICS_WEIGHT", "0.15"))
ENTROPY_WEIGHT = float(os.environ.get("LOSS_ENTROPY_WEIGHT", "0.01"))
UNCERTAINTY_WEIGHT = float(os.environ.get("LOSS_UNCERTAINTY_WEIGHT", "0.02"))

EPOCHS = int(os.environ.get("TRAIN_EPOCHS", "5000"))
PATIENCE = int(os.environ.get("EARLY_STOP_PATIENCE", "250"))
MIN_DELTA = float(os.environ.get("EARLY_STOP_MIN_DELTA", "1e-5"))
PRINT_EVERY = int(os.environ.get("TRAIN_PRINT_EVERY", "100"))

best_loss = float("inf")
best_epoch = 0
best_state = None
epochs_without_improvement = 0

progress = tqdm(range(EPOCHS), desc="training", unit="epoch")
for epoch in progress:
    optimizer.zero_grad(set_to_none=True)
    out = model(x)

    value_loss = value_loss_fn(out["value"], y)
    policy_loss = policy_loss_fn(out["policy_logits"], transition_target)
    nonterminal = (1.0 - terminal).unsqueeze(-1)
    dynamics_loss = (dynamics_loss_fn(out["next_features"], next_x) * nonterminal).sum() / nonterminal.sum().clamp_min(1.0)
    probs = torch.softmax(out["policy_logits"], dim=-1)
    entropy = -(probs * torch.log(probs.clamp_min(1e-8))).sum(dim=-1).mean()
    uncertainty_loss = torch.mean(torch.exp(-out["log_variance"]) * (out["value"] - y).pow(2) + out["log_variance"])

    loss = (
        VALUE_WEIGHT * value_loss
        + POLICY_WEIGHT * policy_loss
        + DYNAMICS_WEIGHT * dynamics_loss
        - ENTROPY_WEIGHT * entropy
        + UNCERTAINTY_WEIGHT * uncertainty_loss
    )
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    loss_value = float(loss.detach().cpu())
    if loss_value < best_loss - MIN_DELTA:
        best_loss = loss_value
        best_epoch = epoch + 1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 0 or (epoch + 1) % PRINT_EVERY == 0:
        progress.set_postfix({
            "loss": f"{loss_value:.5f}",
            "v": f"{float(value_loss.detach().cpu()):.4f}",
            "p": f"{float(policy_loss.detach().cpu()):.4f}",
            "dyn": f"{float(dynamics_loss.detach().cpu()):.4f}",
            "best": f"{best_loss:.5f}@{best_epoch}",
            "patience": f"{epochs_without_improvement}/{PATIENCE}",
        })

    if epochs_without_improvement >= PATIENCE:
        progress.set_postfix({
            "loss": f"{loss_value:.5f}",
            "best": f"{best_loss:.5f}@{best_epoch}",
            "patience": f"{epochs_without_improvement}/{PATIENCE}",
            "stopped": "early",
        })
        print(f"early stopping at epoch={epoch + 1}; best={best_loss:.5f}@{best_epoch}")
        break
progress.close()

if best_state is not None:
    model.load_state_dict(best_state)


In [ ]:
OUT = ROOT / "out/value_model.pt"
OUT.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    "model_state_dict": model.state_dict(),
    "model_type": "transformer_rl_complex_loss",
    "input_dim": x.shape[1],
    "policy_dim": TRANSITION_CLASSES,
    "feature_mean": feature_mean_np.astype(np.float32).tolist(),
    "feature_std": feature_std_np.astype(np.float32).tolist(),
    "loss_weights": {
        "value": VALUE_WEIGHT,
        "policy": POLICY_WEIGHT,
        "dynamics": DYNAMICS_WEIGHT,
        "entropy": ENTROPY_WEIGHT,
        "uncertainty": UNCERTAINTY_WEIGHT,
    },
    "device_used": str(DEVICE),
    "data_path": str(DATA_PATH) if DATA_PATH else None,
}, OUT)
print("saved", OUT)
